# Links Analysis

### Import

In [86]:
from networkx import hits
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

### Fonctions

Matrice d'adjacence

In [131]:
def create_numpy_adjacency_matrix(gml_file):
    # Lire le fichier avec un encodage explicite
    with open(gml_file, 'r', encoding='utf-8') as f:
        graph = nx.parse_gml(f)  # Utiliser parse_gml() au lieu de read_gml()
    adjacency_matrix = nx.to_numpy_array(graph)
    return adjacency_matrix, list(graph.nodes())


Voisins communs

In [132]:
def voisins_communs(matrix):
    # Initialiser une matrice pour stocker les voisins communs
    voisins = np.zeros(matrix.shape)
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            voisins[i, j] = np.sum(np.multiply(matrix[i, :], matrix[j, :]))
    return voisins

Matrice d'attachement préférentiel

In [133]:
def preferential_attachment(matrix):
    degrees = np.sum(matrix, axis=1)
    pref_attach = np.zeros(matrix.shape) 
    # Parcourir les paires de nœuds
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            # L'attachement préférentiel entre deux nœuds est le produit de leurs degrés
            pref_attach[i, j] = degrees[i] * degrees[j]
    return pref_attach 

Matrice de similarité par cosinus

In [134]:
def cosine_similarity(matrix):
    cosine_sim = np.zeros(matrix.shape)
    # Parcourir les paires de nœuds
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            # Calculer le produit scalaire des deux vecteurs correspondant aux nœuds i et j
            dot_product = np.dot(matrix[i, :], matrix[j, :])
            # Calculer les normes des vecteurs pour normalisation
            norm_i = np.linalg.norm(matrix[i, :])
            norm_j = np.linalg.norm(matrix[j, :])         
            # Calculer la similarité cosinus (éviter les divisions par zéro)
            if norm_i > 0 and norm_j > 0:
                cosine_sim[i, j] = dot_product / (norm_i * norm_j)
            else:
                cosine_sim[i, j] = 0  # Si un vecteur a une norme nulle, la similarité est 0
    return cosine_sim

Matrice de similarité par Jaccard

In [135]:
def jaccard_similarity(matrix):
    n = matrix.shape[0]  # Nombre de nœuds
    degrees = np.sum(matrix, axis=1)  # Degré de chaque nœud
    sim_common = np.dot(matrix, matrix.T)  # Nombre de voisins communs entre chaque paire de nœuds
    sim_jac = np.zeros((n, n))  
    # Parcourir chaque paire de nœuds
    for i in range(n):
        for k in range(n):
            # Calculer la mesure de Jaccard (éviter la division par zéro)
            denominator = degrees[i] + degrees[k] - sim_common[i, k]
            if denominator > 0:
                sim_jac[i, k] = sim_common[i, k] / denominator
            else:
                sim_jac[i, k] = 0   
    return sim_jac

In [136]:
def top_20_jaccard_relations(matrix, nodes):
    # Créer une liste pour stocker les relations
    relations = []
    
    # Parcourir la matrice pour trouver les relations
    for i in range(len(matrix)):
        for j in range(len(matrix)):
            if i != j and matrix[i, j] < 0.65:
                relations.append((nodes[i], nodes[j], matrix[i, j]))
    
    # Trier les relations par similarité décroissante
    relations = sorted(relations, key=lambda x: x[2], reverse=True)
    
    # Sélectionner les 20 premières relations
    top_20 = relations[:20]
    
    return top_20

Matrice de Katz

In [137]:
def Katz(graph, alpha):
    #Calculer la matrice de Katz
    A = graph
    n = A.shape[0] # nombre de noeuds
    I = np.identity(n) # matrice identité
    Katz = np.linalg.inv(I - alpha*A) - I # matrice de Katz, on calcule d'abord l'inverse de I - alpha*A puis on soustrait I
    print(f"Ceci est la matrice de Katz :\n {Katz}\n")
    return Katz

Afficher les noeuds avec un score Katz élevé

In [138]:
def identify_indirect_influencers(adjacency_matrix, katz_matrix):
    degree = np.sum(adjacency_matrix, axis=1)
    katz_scores = np.sum(katz_matrix, axis=1)
    
    # Normaliser les valeurs pour comparaison
    scaler = MinMaxScaler()
    degree_scaled = scaler.fit_transform(degree.reshape(-1, 1))
    katz_scaled = scaler.fit_transform(katz_scores.reshape(-1, 1))
    
    # Calculer l'écart entre l'influence directe et indirecte
    influence_gap = katz_scaled - degree_scaled
    
    # Identifier les nœuds avec une influence indirecte forte
    indirect_influencers = np.where((degree_scaled < 0.3) & (katz_scaled > 0.4))[0]
    
    return indirect_influencers, influence_gap

Probabilité de transition matrice

In [139]:
def transition_proba(graph):
    A = graph
    n = A.shape[0] # nombre de noeuds
    degrees = [sum(A[i]) for i in range(n)] # degré de chaque noeud
    transition_graph = np.zeros((n,n))
    """
    au début je n'avais pas mis (transition_graph) mais c'est nécessaire sinon on modifie la matrice A pendant
    le calcul et on obtient un matrice null
    """

    #Calculer la matrice de transition
    for i in range(len(A)):
        for k in range(len(A)):
            transition_graph[i,k] = A[i,k]/degrees[i]
    return transition_graph

Matrice des probas de transition avec ID en etiquettes

In [140]:
def export_edges_with_ids(graph, adjacency_matrix):
    # Extraire les IDs des nœuds
    nodes = list(graph.nodes)  # Récupère les identifiants des nœuds

    # Calculer les degrés sortants
    degrees = np.sum(adjacency_matrix, axis=1)

    # Calculer la matrice des probabilités de transition
    transition_matrix = np.zeros(adjacency_matrix.shape)
    for i in range(adjacency_matrix.shape[0]):
        if degrees[i] > 0:  # Éviter division par zéro
            transition_matrix[i, :] = adjacency_matrix[i, :] / degrees[i]

    # Créer une liste d'arêtes pondérées avec IDs
    edges = []
    for i in range(len(nodes)):
        for j in range(len(nodes)):
            if transition_matrix[i, j] > 0:  # Ignorer les poids nuls
                edges.append([i, j, transition_matrix[i, j]])  # Utilise les indices comme IDs

    # Créer un DataFrame pour l'export
    df_edges = pd.DataFrame(edges, columns=['Source', 'Target', 'Weight'])
    df_edges.to_csv('outputLA/edges_with_ids.csv', index=False, encoding='utf-8')

    print("Exporté : edges_with_ids.csv")

Matrice FPT et CT

In [141]:
def matrice_FPT_et_CT(transition_matrix):
	n = transition_matrix.shape[0]
	FPT = np.zeros((n, n))
	CT = np.zeros((n, n))
	
	for i in range(n):
		for j in range(n):
			if i != j:
				FPT[i, j] = 1 / transition_matrix[i, j] if transition_matrix[i, j] != 0 else np.inf
				CT[i, j] = FPT[i, j] + FPT[j, i]
	
	return FPT, CT

In [142]:
def fpt_ct_between_nodes(FPT, CT, node1, node2):

    fpt_value = FPT[node1, node2]
    ct_value = CT[node1, node2]
    
    return fpt_value, ct_value

Score Hub

In [143]:
def scores_Hub_Authority(graph):
    # Calculer les scores HITS
    hub_scores, authority_scores = nx.hits(graph)
    
    # Afficher les résultats
    print(f"Ceci est le score de Hub :\n {hub_scores}\n")
    return hub_scores, authority_scores

### Applications

In [130]:
gml_file = 'outputgml/graph.gml'

# Créer la matrice d'adjacence
adjacency_matrix, nodes = create_numpy_adjacency_matrix(gml_file)

# Enregistrer la matrice d'adjacence dans un fichier
np.savetxt('outputLA/adjacency_matrix.txt', adjacency_matrix)

In [ ]:
cosine_sim = cosine_similarity(adjacency_matrix)
print(cosine_sim)

# Enregistrer la matrice de similarité cosinus dans un fichier
np.savetxt('outputLA/cosine_similarity_matrix.txt', cosine_sim)

In [ ]:
jaccard_sim = jaccard_similarity(adjacency_matrix)
print(jaccard_sim)

# Enregistrer la matrice de similarité Jaccard dans un fichier
np.savetxt('outputLA/jaccard_similarity_matrix.txt', jaccard_sim)

In [ ]:
# Charger la matrice de similarité Jaccard depuis le fichier
jaccard_sim_matrix = np.loadtxt("outputLA/jaccard_similarity_matrix.txt")

# Afficher les 20 résultats les plus grands inférieurs à 1
top_20_jaccard = top_20_jaccard_relations(jaccard_sim_matrix, nodes)
for relation in top_20_jaccard:
    print(relation)

In [ ]:
# Déterminer la valeur de alpha
alpha = 0.1

# Calculer la matrice de Katz
katz_matrix = Katz(adjacency_matrix, alpha)

# Enregistrer la matrice de Katz dans un fichier
np.savetxt('outputLA/katz_matrix.txt', katz_matrix)

In [ ]:
adjacency_matrix = np.loadtxt('outputLA/adjacency_matrix.txt')  # Matrice d'adjacence
katz_matrix = np.loadtxt('outputLA/katz_matrix.txt')
degree = np.sum(adjacency_matrix, axis=1) 
katz_scores = np.sum(katz_matrix, axis=1) 
# Utiliser la fonction pour identifier les influenceurs indirects
indirect_influencers, influence_gap = identify_indirect_influencers(adjacency_matrix, katz_matrix)

# Afficher les résultats
print("Nœuds avec une forte influence indirecte :", indirect_influencers)


In [ ]:
pref_attach = preferential_attachment(adjacency_matrix)
print(pref_attach)

# Enregistrer la matrice d'attachement préférentiel dans un fichier
np.savetxt('outputLA/preferential_attachment_matrix.txt', pref_attach)

In [ ]:
transition_graph = transition_proba(adjacency_matrix)
print(transition_graph)

# Enregistrer la matrice de probabilité de transition dans un fichier
np.savetxt('outputLA/transition_probability_matrix.txt', transition_graph)

In [ ]:
# Calculer et enregistrer les matrices FPT et CT
# Exemple d'utilisation
FPT, CT = matrice_FPT_et_CT(transition_graph)
np.savetxt('outputLA/FPT_matrix.txt', FPT)
np.savetxt('outputLA/CT_matrix.txt', CT)

In [ ]:
# Exemple d'utilisation
node1 = 1056
node2 = 1109

fpt_value, ct_value = fpt_ct_between_nodes(FPT, CT, node1, node2)
print(f"FPT entre les nœuds {node1} et {node2} : {fpt_value}")
print(f"CT entre les nœuds {node1} et {node2} : {ct_value}")

In [129]:
with open('outputgml/graph.gml', 'r', encoding='utf-8') as f:
    graph = nx.parse_gml(f)

# Calcul des scores de Hub et Authority
hub_scores, authority_scores = nx.hits(graph)

# Enregistrement des scores dans des fichiers texte
np.savetxt('outputLA/hub_scores.txt', list(hub_scores.values()))
np.savetxt('outputLA/authority_scores.txt', list(authority_scores.values()))